In [19]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csgraph
from sklearn.manifold import SpectralEmbedding
from sklearn.preprocessing import normalize

import numpy as np
import torch
from PIL import Image
import os
import pandas as pd

#importing the used models
from transformers import BlipProcessor, BlipModel

In [42]:
def load_and_filter_data(caption_csv_path):
    img_dir="./images"
    img_prefix="img_"
    df = pd.read_csv(caption_csv_path, encoding='latin1')
    df['image_filename'] = [f"{img_prefix}{i}.jpg" for i in df.index]
    df['full_image_path'] = [f"{img_dir}/{fname}" for fname in df['image_filename']]
    df_valid = df[df['full_image_path'].apply(os.path.exists)].reset_index(drop=True)
    print(f"Valid images with captions: {len(df_valid)}")
    return df_valid


def generate_image_embeddings(image_paths, processor, model, device):
    embeddings = []
    for path in image_paths:
        try:
            image = Image.open(path).convert("RGB")
            inputs = processor(images=image, return_tensors="pt").to(device)
            with torch.no_grad():
                features = model.get_image_features(**inputs)
            features = features / features.norm(p=2, dim=-1, keepdim=True)
            embeddings.append(features.cpu().numpy().squeeze())
        except Exception as e:
            print(f"Error processing image {path}: {e}")
    return np.array(embeddings)

def generate_text_embeddings(texts, processor, model, device):
    embeddings = []
    for text in texts:
        try:
            inputs = processor(text=text, return_tensors="pt").to(device)
            with torch.no_grad():
                features = model.get_text_features(**inputs)
            features = features / features.norm(p=2, dim=-1, keepdim=True)
            embeddings.append(features.cpu().numpy().squeeze())
        except Exception as e:
            print(f"Error processing text '{text[:30]}...': {e}")
    return np.array(embeddings)

def match_embeddings(image_embeddings, text_embeddings, valid_image_paths, valid_captions):
    similarity_matrix = cosine_similarity(image_embeddings, text_embeddings)
    matched_pairs = []
    for i in range(similarity_matrix.shape[0]):
        j = np.argmax(similarity_matrix[i])
        matched_pairs.append({
            "image_path": valid_image_paths[i],
            "caption": valid_captions[j],
            "image_embedding": image_embeddings[i],
            "text_embedding": text_embeddings[j],
            "similarity": similarity_matrix[i, j]
        })
    print(f"Matched {len(matched_pairs)} image-caption pairs via cosine similarity.")
    return matched_pairs, similarity_matrix

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipModel.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

df_long = load_and_filter_data("./long_generated_captions.csv")
df_short = load_and_filter_data("./short_generated_captions.csv")

valid_image_paths = df_long['full_image_path'].tolist()
valid_captions_long = df_long['paraphrased_caption'].tolist()
valid_captions_short = df_short['paraphrased_caption'].tolist()

image_embeddings = generate_image_embeddings(valid_image_paths, processor, model, device)
text_embeddings_long = generate_text_embeddings(valid_captions_long, processor, model, device)
text_embeddings_short = generate_text_embeddings(valid_captions_short, processor, model, device)
matched_pairs_long, similarity_matrix_spectral_long = match_embeddings(image_embeddings, text_embeddings_long, valid_image_paths, valid_captions_long)
matched_pairs_short, similarity_matrix_spectral_short = match_embeddings(image_embeddings, text_embeddings_short, valid_image_paths, valid_captions_short)


`BlipModel` is going to be deprecated in future release, please use `BlipForConditionalGeneration`, `BlipForQuestionAnswering` or `BlipForImageTextRetrieval` depending on your usecase.
Some weights of BlipModel were not initialized from the model checkpoint at Salesforce/blip-image-captioning-base and are newly initialized: ['logit_scale', 'text_model.embeddings.LayerNorm.bias', 'text_model.embeddings.LayerNorm.weight', 'text_model.embeddings.position_embeddings.weight', 'text_model.embeddings.word_embeddings.weight', 'text_model.encoder.layer.0.attention.output.LayerNorm.bias', 'text_model.encoder.layer.0.attention.output.LayerNorm.weight', 'text_model.encoder.layer.0.attention.output.dense.bias', 'text_model.encoder.layer.0.attention.output.dense.weight', 'text_model.encoder.layer.0.attention.self.key.bias', 'text_model.encoder.layer.0.attention.self.key.weight', 'text_model.encoder.layer.0.attention.self.query.bias', 'text_model.encoder.layer.0.attention.self.query.weight', 'text_mo

Valid images with captions: 498
Valid images with captions: 498


In [23]:
def apply_spectral_embedding_from_cosine(B, n_components=128):
    B = np.clip(B, 0, None)

    n_texts, n_images = B.shape

    zero_text = np.zeros((n_texts, n_texts))
    zero_image = np.zeros((n_images, n_images))
    
    A = np.block([
        [zero_text, B],
        [B.T, zero_image]
    ])

    epsilon = 1e-3
    A += epsilon
    np.fill_diagonal(A, epsilon)

    n_total = A.shape[0]
    if n_components >= n_total:
        n_components = n_total - 1
        print(f"Warning: n_components reduced to {n_components} because it must be less than total nodes {n_total}")

    spectral = SpectralEmbedding(n_components=n_components, affinity='precomputed')
    embedded = spectral.fit_transform(A)

    spectral_text = normalize(embedded[:n_texts])
    spectral_image = normalize(embedded[n_texts:])
    
    return spectral_image, spectral_text


In [24]:
spectral_image_embeddings, spectral_text_long_embeddings = apply_spectral_embedding_from_cosine(similarity_matrix_spectral_long)
spectral_image_embeddings, spectral_text_short_embeddings = apply_spectral_embedding_from_cosine(similarity_matrix_spectral_short)